In [ ]:
import pandas as pd
import numpy as np
import default_risk.config as cfg
from dotenv import load_dotenv


previous_application_df = pd.read_parquet(cfg.CLEANS_DIR / "previous_application_train-cleaned.parquet")



In [ ]:
previous_application_df.head(5)

In [ ]:
previous_application_df = pd.read_parquet(cfg.CLEANS_DIR / "previous_application_train-cleaned.parquet")

#creating columns before aggregation
previous_application_df["diff_application_credit"] = previous_application_df["amt_application"] - previous_application_df["amt_credit"]
previous_application_df["ratio_credit_to_goods"] = previous_application_df["amt_credit"] / (previous_application_df["amt_goods_price"].replace(0,np.nan))
previous_application_df["total_interest_charged"] = ((previous_application_df["amt_annuity"] * previous_application_df["cnt_payment"]) - previous_application_df["amt_credit"]).clip(lower=0)
previous_application_df["implied_interest_rate"] = (previous_application_df["amt_annuity"] * previous_application_df["cnt_payment"]) / previous_application_df["amt_credit"].replace(0, np.nan)
previous_application_df["ratio_credit_to_annuity"]= previous_application_df["amt_credit"] / (previous_application_df["amt_annuity"].replace(0,np.nan))

previous_application_df["log_amt_credit"] = np.log1p(previous_application_df["amt_credit"])
previous_application_df["log_amt_application"] = np.log1p(previous_application_df["amt_application"])
previous_application_df["log_amt_annuity"] = np.log1p(previous_application_df["amt_annuity"])
previous_application_df["log_amt_down_payment"] = np.log1p(previous_application_df["amt_down_payment"])
previous_application_df["log_amt_goods_price"] = np.log1p(previous_application_df["amt_goods_price"])
previous_application_df["log_diff_application_credit"] = previous_application_df["log_amt_application"] - previous_application_df["log_amt_credit"]
previous_application_df["log_total_interest_charged"] = np.log1p((previous_application_df["total_interest_charged"]))

#we will calculate aggregations per client for differents tables so we will separate dictionaries per table
agg_from_prev_app_dict= {
    #saving the ammount of contract
    "id_prev" : ["count"],
    
    #for log transformated we want to catch the mean and the std (avoiding the impact of the heavy tail from this columns)
    "log_amt_credit": ["mean","std"],   
    "log_amt_application": ["mean","std"],
    "log_amt_down_payment": ["mean","std"],
    "log_amt_goods_price": ["mean","std"],
    "log_amt_annuity": ["mean","std"],
    "log_total_interest_charged" : ["mean","std"],
    "instalments_completion_ratio" : ["mean","std"],
 
    #for non transformated columns we want to catch the representative values and the acumulated
    "amt_credit": ["max", "min","median","sum"],
    "amt_application": ["max", "min","median","sum"],
    "amt_down_payment": ["max", "min","median","sum"],
    "amt_goods_price": ["max", "min","median","sum"],
    "amt_annuity": ["max", "min","median"],
    "total_interest_charged": ["max", "min","median"],
    "implied_interest_rate" : ["max", "min","mean","std"],
    
    #others_monetary
    "diff_application_credit": ["max","mean","min","sum","median"], #
    "log_diff_application_credit": ["max","mean","min"],
    "rate_down_payment": ["max","mean","min","median","std"], #
    "ratio_credit_to_goods" : ["max","mean","median","min","std"], # 
    "ratio_credit_to_annuity" : ["max","mean","min","median","std"], #
    #categoricals
    "name_contract_status_canceled": ["mean"],#,"sum"
    "name_contract_status_refused": ["mean"],#,"sum"
    "amt_annuity_and_cnt_payment_are_missing" :["mean","sum"],
    "amt_down_payment_is_missing" : ["mean","sum"],
    "nflag_insured_on_approval" : ["mean","sum"],
    "days_and_insurance_information_are_missing": ["mean","sum"],
    "amt_goods_price_is_missing" : ["mean","sum"],
    "rate_down_payment_is_missing" : ["mean","sum"],


    #counters
    "days_decision":["mean","min","max"],
    "cnt_payment":["mean","max","sum"]
}

final_dict = agg_from_prev_app_dict


In [ ]:
instalament_df = pd.read_parquet(cfg.PROCESSED_DIR / "installments_payments_train-processed.parquet")
previous_application_df= previous_application_df.merge(instalament_df,how="left",on= "id_prev")

agg_from_instalment_payment_dict= {
    "instalments_amount_of_versions_in_sequence" : ["mean","max","sum"],
    "instalments_potentially_on_going" : ["sum"],
    "instalments_dead_tail_length" : ["mean","max"],
    "instalments_amt_instalment_sum" : ["mean","sum","max"],
    "instalments_amt_payment_sum" : ["mean", "sum", "max"],
    "instalments_days_of_delinquency_max": ["max"],
    "instalments_days_of_delinquency_mean": ["mean", "max"],
    "instalments_extra_instalament_sum":["sum"],
    "instalments_extra_instalament_mean":["mean"],
    "instalments_days_in_advance_max":["max"],
    "instalments_days_of_underpayment_max":["max"],
    "instalments_days_in_advance_mean": ["mean", "max"],
    "instalments_is_delinquency_sum" : ["sum"] ,
    "instalments_is_delinquency_mean" : ["mean"],
    "instalments_diff_expected_received_sum" : ["sum"]
} 



final_dict = final_dict | agg_from_instalment_payment_dict

In [ ]:
credit_card_df = pd.read_parquet(cfg.PROCESSED_DIR / "credit_card_balance_train-processed.parquet")
previous_application_df= previous_application_df.merge(credit_card_df,how="left",on= "id_prev")


agg_from_credit_card_dict= {
    "credit_card_desesperation_ratio_max" : ["max"],
    "credit_card_desesperation_ratio_min" : ["min"],
    "credit_card_desesperation_ratio_mean"  : ["mean"],
    "credit_card_balance_limit_ratio_min" : ["min"],
    "credit_card_payment_ratio_min" : ["min"],
    "credit_card_payment_ratio_mean" : ["mean"],
    "credit_card_potential_on_going_loan" : ["sum"],
    "credit_card_is_over_the_limit_sum" : ["sum"],
    "credit_card_is_over_the_limit_mean" : ["mean"],
    "credit_card_months_balance_min" : ["min"],
    "credit_card_sk_dpd_max" : ["max"],
    "credit_card_sk_dpd_def_max" : ["max"],
    "credit_card_inconsistency_gap_max" : ["max"],
    "credit_card_cnt_instalment_mature_cum_max" : ["max"],
    "credit_card_cnt_drawings_atm_current_mean": ["mean"],
    "credit_card_amt_balance_mean" : ["mean"],
    "credit_card_amt_balance_max" : ["max"],
}

final_dict = final_dict | agg_from_credit_card_dict

In [ ]:
cash_balance_f = pd.read_parquet(cfg.PROCESSED_DIR / "POS_CASH_balance_train-processed.parquet")
previous_application_df= previous_application_df.merge(cash_balance_f,how="left",on= "id_prev")

agg_from_cash_balance_dict= {
    "cash_balance_potentaily_on_going" : ["sum"],
    "cash_balance_diff_expected_real_duration" : ["mean"],
    "cash_balance_sk_dpd_mean"  : ["mean"],
    "cash_balance_original_expected_duration" : ["max"],
    "cash_balance_sk_dpd_def_mean" : ["mean"],
    "cash_balance_factical_duration" : ["mean"],
    "cash_balance_sk_dpd_tecnical_mean": ["mean"],
    "cash_balance_sk_dpd_tecnical_sum": ["sum"],  
    "cash_balance_sk_dpd_severe_mean": ["mean"], 
    "cash_balance_sk_dpd_severe_sum": ["mean"], 
    "cash_balance_dpd_def_tecnical_mean": ["mean"], 
    "cash_balance_dpd_def_tecnical_sum": ["sum"], 
    "cash_balance_dpd_def_severe_mean": ["mean"], 
    "cash_balance_dpd_def_severe_sum": ["sum"], 
}

final_dict = final_dict | agg_from_cash_balance_dict



In [ ]:

previous_application_to_pivot_df= previous_application_df.drop(columns="id_prev")
mask_no_final= previous_application_df["flag_last_application_for_the_contract"] == "N"
previous_application_to_pivot_df= previous_application_to_pivot_df.loc[~mask_no_final]
previous_application_to_pivot_df= previous_application_to_pivot_df.drop(columns=["flag_last_application_for_the_contract"])

previous_application_to_pivot_df.sort_values(["id_curr","days_decision"],inplace=True,ascending=False)
last_three= previous_application_to_pivot_df.groupby("id_curr").head(1)
last_three = last_three.copy()
last_three["loan_order"] = last_three.groupby("id_curr").cumcount() + 1
df_wide = last_three.pivot(index="id_curr", columns="loan_order")
df_wide.columns =[f"{col}_prev_{rank}" for col, rank in df_wide.columns]
df_wide= df_wide.reset_index()
previous_application_df["code_reject_reason"]= previous_application_df["code_reject_reason"].str.lower()
previous_application_df = pd.get_dummies(previous_application_df, columns=[ "code_reject_reason"])
previous_application_df["name_contract_status"]= previous_application_df["name_contract_status"].str.lower()
previous_application_df = pd.get_dummies(previous_application_df, columns=[ "name_contract_status"])
agg_metrics_df= previous_application_df.groupby("id_curr").agg(final_dict)

agg_metrics_df.columns= [f"{col[0]}_{col[1]}" for col in agg_metrics_df.columns]

agg_metrics_df.rename(columns={"id_prev_count": "applications_count"},inplace=True)
agg_metrics_df= agg_metrics_df.reset_index()

agg_metrics_df["global_approval_ratio"] = agg_metrics_df["amt_credit_sum"] / agg_metrics_df["amt_application_sum"].replace(0, np.nan)
agg_metrics_df["days_decision_spread"] = agg_metrics_df["days_decision_max"] - agg_metrics_df["days_decision_min"]

previous_application_ready_to_merge= df_wide.merge(agg_metrics_df,on="id_curr",how="left")


In [ ]:
instalament_time_window_df= pd.read_parquet(cfg.PROCESSED_DIR / "installments_payments_train-processed-temporal_window.parquet")
previous_application_ready_to_merge= previous_application_ready_to_merge.merge(instalament_time_window_df,how="left",on="id_curr")


In [ ]:
credit_card_time_window_df= pd.read_parquet(cfg.PROCESSED_DIR / "credit_card_balance_train-processed-last_six_months_agg.parquet")
previous_application_ready_to_merge= previous_application_ready_to_merge.merge(credit_card_time_window_df,how="left",on="id_curr")


In [ ]:
cash_balance_time_window_df= pd.read_parquet(cfg.PROCESSED_DIR / "POS_CASH_balance_train-processed_time_window.parquet")
previous_application_ready_to_merge= previous_application_ready_to_merge.merge(cash_balance_time_window_df,how="left",on="id_curr")


In [ ]:
previous_application_ready_to_merge.to_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet", index=False)